In [ ]:
# Install
%pip install torch
%pip install torchvision

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

from MLP import FashionMLP
from Part1 import train

In [ ]:
# Set seed
seed = 42
torch.manual_seed(seed)
# Set device to GPU if available, otherwise CPU
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed(seed)
print(f"Using device: {device}")

In [ ]:
# Préparation de données

# Normalement on aurait calculé la moyenne et l'écart-type de Fashion-MNIST, mais ici ils sont donnés
mean = 0.2860
std = 0.3530

# Voici la transformé pour expérience 2
'''
transform = transforms.Compose([
    transforms.Lambda(lambda img: torch.from_numpy(np.array(img)).float())
])
'''

# Voici la transformé pour le reste des expériences
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])



def get_loaders(batch_size, transform):
    os.makedirs("./data", exist_ok=True)

    # Download and load training data
    train_dataset = torchvision.datasets.FashionMNIST(
        root='./data', train=True, transform=transform, download=True
    )

    # Download and load test data
    test_dataset = torchvision.datasets.FashionMNIST(
        root='./data', train=False, transform=transform, download=True
    )
    train_loader = torch.utils.data.DataLoader(
        dataset=train_dataset, batch_size=batch_size, shuffle=True
    )
    test_loader = torch.utils.data.DataLoader(
        dataset=test_dataset, batch_size=batch_size, shuffle=False
    )
    return train_loader, test_loader


In [ ]:
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

In [ ]:
# Voyons un peu de quoi a l'air le jeu de données
train_dataset = torchvision.datasets.FashionMNIST(
        root='./data', train=True, transform=transform, download=True
    )

np.set_printoptions(linewidth=350)

image_tensor, label_idx = train_dataset[0]
image_numpy = image_tensor.squeeze().numpy()
fig = plt.figure()
plt.imshow(image_numpy, cmap='gray')
plt.title(class_names[label_idx])
plt.tight_layout()
plt.show()
print(image_tensor.numpy())

In [ ]:
# Voici un fonction pour visualiser quelques exemples
def visualize_samples(dataset, num_samples=10):
    """
    Plots a grid of random images from the dataset with their labels.
    """
    fig = plt.figure(figsize=(12, 5))

    for i in range(num_samples):
        idx = np.random.randint(0, len(dataset))
        image_tensor, label_idx = dataset[idx]
        image_numpy = image_tensor.squeeze().numpy()
        ax = fig.add_subplot(2, 5, i + 1, xticks=[], yticks=[])
        ax.imshow(image_numpy, cmap='gray')
        ax.set_title(class_names[label_idx])
    plt.tight_layout()
    plt.show()

In [ ]:
print(f"Total number of training images: {len(train_dataset)}")
print("Displaying random samples...")

visualize_samples(train_dataset, num_samples=10)

In [ ]:
# Hyperparamètres d'entraînement
BATCH_SIZE = 64
LEARNING_RATE = 0.05
EPOCHS = 30

In [ ]:
# Hyperparamètres du modèle
INPUT_SIZE = 28*28
H1_SIZE = 64
H2_SIZE = 64
OUTPUT_SIZE = 10

ACTIVATION1 = nn.Sigmoid()
ACTIVATION2 = nn.Sigmoid()

In [ ]:
model = FashionMLP(input_size=INPUT_SIZE, h1_size=H1_SIZE, h2_size=H2_SIZE, output_size=OUTPUT_SIZE, activation1=ACTIVATION1, activation2=ACTIVATION2).to(device)
print(model)

In [ ]:
def mse_los(outputs, labels):
    targets_one_hot = F.one_hot(labels, num_classes=outputs.size(1)).float()
    probs = F.softmax(outputs, dim=1)
    return F.mse_loss(probs, targets_one_hot)

def hinge_loss_simple(outputs, labels, margin=1.0):
    correct_scores = outputs.gather(1, labels.view(-1, 1))
    margins = outputs - correct_scores + margin
    loss = torch.max(torch.zeros_like(margins), margins)
    loss.scatter_(1, labels.view(-1, 1), 0)
    return loss.sum(1).mean()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE) # Nous utilisons SGD tel que vu en classe. Ne changer pas l'optimizer
train_loader, test_loader = get_loaders(BATCH_SIZE)

In [ ]:
train_losses, test_losses, train_accuracies, test_accuracies = train(model=model, epochs=EPOCHS, optimizer=optimizer, criterion=criterion, train_loader=train_loader, test_loader=test_loader, device=device)

In [ ]:
# Visualisation
def visualize(experiment, experiment_name, epochs):
  plt.figure(figsize=(12, 5))

  # Plot training loss
  plt.subplot(1, 2, 1)
  plt.plot(range(1, epochs + 1), experiment[0], marker='o', color='orange', label=f'{experiment_name} Train Loss')
  plt.plot(range(1, epochs + 1), experiment[1], marker='x', ls="--", color='orange', label=f'{experiment_name} Test Loss')
  plt.title('Training Loss over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.grid(True)
  plt.legend()

  # Plot test accuracy
  plt.subplot(1, 2, 2)
  plt.plot(range(1, epochs + 1), experiment[2], marker='o', color='orange', label=f'{experiment_name} train Accuracy')
  plt.plot(range(1, epochs + 1), experiment[3], marker='x', ls="--", color='orange', label=f'{experiment_name} Test Accuracy')
  plt.title('Test Accuracy over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy (%)')
  plt.legend()
  plt.grid(True)

  plt.tight_layout()
  plt.savefig(f'exp_{experiment_name}.png')
  plt.show()

In [ ]:
# Une fonction sympa pour voir les exemples mal classifiés
def get_model_failures(model, loader, device):
    """
    Runs the model on the loader and returns a list of (image, true_label, predicted_label)
    for every instance where the model was wrong.
    """
    model.eval()
    failures = []

    print("Analyzing test set for failures...")
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            mask = predicted != labels
            wrong_imgs = images[mask].cpu()
            wrong_labels = labels[mask].cpu()
            wrong_preds = predicted[mask].cpu()

            for i in range(len(wrong_imgs)):
                failures.append((wrong_imgs[i], wrong_labels[i].item(), wrong_preds[i].item()))

    print(f"Analysis complete. Model failed on {len(failures)} images.")
    return failures

def visualize_failures(failures, class_names, num_samples=10):
    """
    Specialized visualization for failures to show what the model
    thought the image was versus reality.
    """
    if not failures:
        print("No failures to display!")
        return

    num_samples = min(num_samples, len(failures))
    fig = plt.figure(figsize=(15, 7))

    indices = np.random.choice(len(failures), num_samples, replace=False)

    for i, idx in enumerate(indices):
        image_tensor, true_label, pred_label = failures[idx]

        image_numpy = image_tensor.squeeze().numpy()

        ax = fig.add_subplot(2, 5, i + 1, xticks=[], yticks=[])
        ax.imshow(image_numpy, cmap='gray')

        title = f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}"
        ax.set_title(title, fontsize=10)

    plt.tight_layout()
    plt.suptitle("Model Misclassifications", fontsize=16, y=1.05)
    plt.show()

In [ ]:
failed_data = get_model_failures(model, test_loader, device)

In [ ]:
visualize_failures(failed_data, class_names, num_samples=10)